# TOKEN CLASSIFICATION (CHUNKING)

In [1]:
import os
import shutil
import sys

print("Phase 1: Cleaning up broken environment files...")
bad_folder = r"C:\Users\bhuva\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision"

if os.path.exists(bad_folder):
    try:
        shutil.rmtree(bad_folder)
        print("✅ Successfully deleted the corrupted torchvision folder!")
    except Exception as e:
        print(f"⚠️ Could not delete folder (it might already be gone): {e}")
else:
    print("✅ No corrupted folders found.")

print("\nPhase 2: Forcing installation of required NLP libraries...")
# Remove conflicting vision libraries
!{sys.executable} -m pip uninstall -y torchvision ultralytics

# Install exact versions needed for text processing
!{sys.executable} -m pip install transformers[torch] accelerate datasets evaluate seqeval scikit-learn --upgrade

print("\n🎉 Setup Complete! STOP HERE AND RESTART YOUR KERNEL.")

Phase 1: Cleaning up broken environment files...
✅ No corrupted folders found.

Phase 2: Forcing installation of required NLP libraries...



🎉 Setup Complete! STOP HERE AND RESTART YOUR KERNEL.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\bhuva\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


In [4]:
# ==============================================================================
# ASSIGNMENT NLP-5: TOKEN CLASSIFICATION (CHUNKING) 
# Task Description: Fine-tune DistilBERT for Chunking using CoNLL-2003
# ==============================================================================

# --- SUPPRESS WARNINGS FOR A CLEAN NOTEBOOK OUTPUT ---
import warnings
warnings.filterwarnings("ignore")
import logging
logging.getLogger("transformers").setLevel(logging.ERROR)
# -----------------------------------------------------

import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorForTokenClassification
)
import evaluate

# Setup Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# ---------------------------------------------------------
# Task 1: Dataset Selection
# ---------------------------------------------------------
print("\nLoading CoNLL-2003 Dataset...")
dataset = load_dataset("conll2003", trust_remote_code=True)

# Using a subset of the data so your local CPU can finish training
train_dataset = dataset["train"].select(range(2000)) 
val_dataset = dataset["validation"].select(range(500))

chunk_tags = dataset["train"].features["chunk_tags"].feature.names
num_labels = len(chunk_tags)

id2label = {i: label for i, label in enumerate(chunk_tags)}
label2id = {label: i for i, label in enumerate(chunk_tags)}

print(f"Dataset Selected: CoNLL-2003")
print(f"Number of Chunk Tags: {num_labels}")

# ---------------------------------------------------------
# Task 2: Data Preprocessing
# ---------------------------------------------------------
print("\nInitializing Tokenizer and Aligning Labels...")
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples["chunk_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i) 
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_train = train_dataset.map(tokenize_and_align_labels, batched=True)
tokenized_val = val_dataset.map(tokenize_and_align_labels, batched=True)

# ---------------------------------------------------------
# Task 3: Model Setup
# ---------------------------------------------------------
print("\nSetting up DistilBERT Model...")
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

# ---------------------------------------------------------
# Task 5: Evaluation Setup
# ---------------------------------------------------------
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [chunk_tags[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [chunk_tags[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    # zero_division=0 prevents the pink warning during epoch 1
    results = seqeval.compute(predictions=true_predictions, references=true_labels, zero_division=0)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# ---------------------------------------------------------
# Task 4: Training Setup
# ---------------------------------------------------------
print("\nStarting Training Pipeline...")

training_args = TrainingArguments(
    output_dir="./chunking_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch", 
    save_strategy="epoch",
    report_to="none", 
    disable_tqdm=True,
    dataloader_pin_memory=False # <--- Removes the PyTorch pink warning!
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer, 
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Training Model... (Output will be clean and neat!)")
trainer.train()

# ---------------------------------------------------------
# Task 6: Inference
# ---------------------------------------------------------
print("\nPerforming Inference...")

from transformers import pipeline
chunker = pipeline("token-classification", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

test_sentence = "John works at Google in California" 
predictions = chunker(test_sentence)

print(f"\nInput Sentence: '{test_sentence}'\n")
print("Predicted Chunk Tags:")
for pred in predictions:
    print(f"Word: {pred['word']:<12} | Chunk: {pred['entity_group']}")

# ---------------------------------------------------------
# Task 7 & 8: Comparison and Report
# ---------------------------------------------------------
print("\n" + "="*50)
print("Task 7 & 8: Analysis and Report")
print("="*50)
print("""
Comparison: POS Tagging vs. Chunking
- POS Tagging (Grammar-level): Identifies the basic grammatical role of each individual word (e.g., Noun, Verb, Adjective). It focuses on single tokens.
- Chunking (Phrase-level): Groups words into meaningful phrases (e.g., Noun Phrases, Verb Phrases). It builds upon POS tags to understand sentence structure.

Challenges Faced:
1. Subword Tokenization: Transformers split words like 'running' into 'run' and '##ning'. The challenge is ensuring only the first subword gets the label, and subsequent subwords are ignored (label -100) to avoid confusing the loss calculation.
2. Label Alignment: Keeping the labels synchronized with the newly tokenized input_ids requires careful manipulation of the word_ids list.

Observations and Insights:
- The `seqeval` metric is essential for token classification tasks as it evaluates the predicted sequences as a whole, rather than just individual token accuracy.
- Fine-tuning DistilBERT yields good results even on a subset of data, making it a highly efficient model for local sequence labeling.
""")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'conll2003' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Using device: cpu

Loading CoNLL-2003 Dataset...


Using the latest cached version of the dataset since conll2003 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'conll2003' at C:\Users\bhuva\.cache\huggingface\datasets\conll2003\conll2003\1.0.0\9a4d16a94f8674ba3466315300359b0acd891b68b6c8743ddf60b9c702adce98 (last modified on Sun Apr  5 09:04:40 2026).


Dataset Selected: CoNLL-2003
Number of Chunk Tags: 23

Initializing Tokenizer and Aligning Labels...

Setting up DistilBERT Model...


Loading weights: 100%|█████████████████████████████████████████████████████████████| 100/100 [00:00<00:00, 5202.43it/s]


...

{'eval_loss': '0.3166', 'eval_precision': '0.8429', 'eval_recall': '0.8352', 'eval_f1': '0.839', 'eval_accuracy': '0.9211', 'eval_runtime': '21.4', 'eval_samples_per_second': '23.37', 'eval_steps_per_second': '1.495', 'epoch': '2'}


Writing model shards: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.16it/s]


{'eval_loss': '0.3093', 'eval_precision': '0.8568', 'eval_recall': '0.8403', 'eval_f1': '0.8485', 'eval_accuracy': '0.9244', 'eval_runtime': '19.44', 'eval_samples_per_second': '25.73', 'eval_steps_per_second': '1.646', 'epoch': '3'}


Writing model shards: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.33it/s]


{'train_runtime': '1089', 'train_samples_per_second': '5.512', 'train_steps_per_second': '0.344', 'train_loss': '0.5585', 'epoch': '3'}

Performing Inference...

Input Sentence: 'John works at Google in California'

Predicted Chunk Tags:
Word: john         | Chunk: NP
Word: works        | Chunk: VP
Word: at           | Chunk: PP
Word: google       | Chunk: NP
Word: in           | Chunk: PP
Word: california   | Chunk: NP

Task 7 & 8: Analysis and Report

Comparison: POS Tagging vs. Chunking
- POS Tagging (Grammar-level): Identifies the basic grammatical role of each individual word (e.g., Noun, Verb, Adjective). It focuses on single tokens.
- Chunking (Phrase-level): Groups words into meaningful phrases (e.g., Noun Phrases, Verb Phrases). It builds upon POS tags to understand sentence structure.

Challenges Faced:
1. Subword Tokenization: Transformers split words like 'running' into 'run' and '##ning'. The challenge is ensuring only the first subword gets the label, and subsequent subwo